In [1]:
from quick_spice_manager import SpiceManager
spice_manager = SpiceManager()

tc = spice_manager.tour_config

In [2]:
from spice_segmenter import PhaseAngle, Distance

start, end = "2033-01-01", "2033-12-31" 

In [3]:
c = PhaseAngle("JUICE_JANUS", "GANYMEDE") > "20 deg"

c2 = Distance("JUICE_JANUS", "GANYMEDE") < "5e4 km"

In [4]:
# Example 1: Serialize a simple constraint
json_str = c.to_json(indent=2)
print("Serialized constraint:")
print(json_str[:300] + "..." if len(json_str) > 300 else json_str)

# Deserialize it back - use Constraint.from_json() for constraints
from spice_segmenter import Constraint
reconstructed = Constraint.from_json(json_str)
print(f"\nReconstructed: {reconstructed}")
print(f"Types match: {type(reconstructed) == type(c)}")

Serialized constraint:
{
  "time_step": null,
  "left": {
    "observer": "JUICE_JANUS",
    "target": "GANYMEDE",
    "light_time_correction": "NONE",
    "third_body": "SUN",
    "type": "PhaseAngle"
  },
  "right": {
    "_value": "20 degree",
    "type": "ScalarConstant"
  },
  "operator": ">",
  "type": "Constraint"
...

Reconstructed: (Phase Angle of GANYMEDE with respect to SUN as seen from JUICE_JANUS > 20 degree)
Types match: True


In [5]:
# Example 2: Serialize a complex nested constraint
cc = c & c2
json_str_nested = cc.to_json(indent=2)
print("Complex constraint JSON (first 400 chars):")
print(json_str_nested[:400] + "...")

# Deserialize
reconstructed_nested = Constraint.from_json(json_str_nested)
print(f"\nReconstructed: {reconstructed_nested}")
print(f"Operator: {reconstructed_nested.operator}")
print(f"Left type: {type(reconstructed_nested.left).__name__}")
print(f"Right type: {type(reconstructed_nested.right).__name__}")

Complex constraint JSON (first 400 chars):
{
  "time_step": null,
  "left": {
    "time_step": null,
    "left": {
      "observer": "JUICE_JANUS",
      "target": "GANYMEDE",
      "light_time_correction": "NONE",
      "third_body": "SUN",
      "type": "PhaseAngle"
    },
    "right": {
      "_value": "20 degree",
      "type": "ScalarConstant"
    },
    "operator": ">",
    "type": "Constraint"
  },
  "right": {
    "time_step": null...

Reconstructed: ((Phase Angle of GANYMEDE with respect to SUN as seen from JUICE_JANUS > 20 degree) & (Distance of GANYMEDE from JUICE_JANUS < 50000.0 kilometer))
Operator: &
Left type: Constraint
Right type: Constraint

{
  "time_step": null,
  "left": {
    "time_step": null,
    "left": {
      "observer": "JUICE_JANUS",
      "target": "GANYMEDE",
      "light_time_correction": "NONE",
      "third_body": "SUN",
      "type": "PhaseAngle"
    },
    "right": {
      "_value": "20 degree",
      "type": "ScalarConstant"
    },
    "operator": "

In [6]:
# Example 3: Serialize a simple Property (not a constraint)
phase_angle = PhaseAngle("JUICE_JANUS", "GANYMEDE")
prop_json = phase_angle.to_json()
print(f"Property JSON: {prop_json}")

# Deserialize - use the specific class
reconstructed_prop = PhaseAngle.from_json(prop_json)
print(f"\nReconstructed property: {reconstructed_prop}")
print(f"Type: {type(reconstructed_prop).__name__}")

Property JSON: {"observer": "JUICE_JANUS", "target": "GANYMEDE", "light_time_correction": "NONE", "third_body": "SUN", "type": "PhaseAngle"}

Reconstructed property: Phase Angle of GANYMEDE with respect to SUN as seen from JUICE_JANUS
Type: PhaseAngle


In [7]:
from spice_segmenter import Constant, Constraint, Property
Property.from_json(prop_json)

Phase Angle of GANYMEDE with respect to SUN as seen from JUICE_JANUS

In [8]:
from spice_segmenter.serialization import create_property_converter
from spice_segmenter import Property
from spice_segmenter.component_selector import ComponentSelector
from spice_segmenter.coordinates import SphericalCoordinates, Vector



v = Vector('JUICE_JANUS', 'GANYMEDE')





conv = create_property_converter()

In [9]:
conv.unstructure(v)

{'frame': 'J2000',
 'abcorr': 'NONE',
 'origin': 'JUICE_JANUS',
 'target': 'GANYMEDE',
 'type': 'Vector'}

In [10]:
raise

RuntimeError: No active exception to reraise

In [ ]:
# c2.solve(start, end)

In [ ]:
cc = c  & c2

In [ ]:
from spice_segmenter import Constant, Constraint, Property


cc.right.left


def collect_properties(constraint):
    props = list()

    if not isinstance(constraint, Constraint):
        print("Not a constraint:", constraint)
        return props
    items = [constraint.right, constraint.left]

    for item in items:
        if isinstance(item, Constraint):
            props.extend(collect_properties(item))
        elif isinstance(item, Constant):
            pass
        else:
            props.append(item)

    return props



used_properties = {}
for p in collect_properties(cc):
    values = {}
    values['description'] = str(p)

    used_properties[p.name] = values
    used_properties[p.name]['class'] = p.__class__.__name__
    used_properties[p.name]['units'] = str(p.unit)



In [ ]:
# Use the serialization module from spice_segmenter
from spice_segmenter import create_property_converter

converter = create_property_converter()

data = converter.unstructure(cc)
data

{'time_step': None,
 'left': {'time_step': None,
  'left': {'observer': 'JUICE_JANUS',
   'target': 'GANYMEDE',
   'light_time_correction': 'NONE',
   'third_body': 'SUN',
   'type': 'PhaseAngle'},
  'right': {'_value': '20 degree', 'type': 'ScalarConstant'},
  'operator': '>',
  'type': 'Constraint'},
 'right': {'time_step': None,
  'left': {'observer': 'JUICE_JANUS',
   'target': 'GANYMEDE',
   'light_time_correction': 'NONE',
   'type': 'Distance'},
  'right': {'_value': '50000.0 kilometer', 'type': 'ScalarConstant'},
  'operator': '<',
  'type': 'Constraint'},
 'operator': '&',
 'type': 'Constraint'}

## Testing the Serialization Module

The `spice_segmenter.serialization` module provides utilities for serializing and deserializing Property and Constraint objects using cattrs with tagged unions.

In [ ]:
converter.structure(data, Constraint) == cc


(((Phase Angle of GANYMEDE with respect to SUN as seen from JUICE_JANUS > 20 degree) & (Distance of GANYMEDE from JUICE_JANUS < 50000.0 kilometer)) = ((Phase Angle of GANYMEDE with respect to SUN as seen from JUICE_JANUS > 20 degree) & (Distance of GANYMEDE from JUICE_JANUS < 50000.0 kilometer)))

In [ ]:
# Verify it worked correctly
restructured = converter.structure(data, Constraint)
print(f"Original: {cc}")
print(f"Restructured: {restructured}")
print(f"\nTypes match: {type(cc) == type(restructured)}")
print(f"Left types match: {type(cc.left) == type(restructured.left)}")
print(f"Right types match: {type(cc.right) == type(restructured.right)}")

Original: ((Phase Angle of GANYMEDE with respect to SUN as seen from JUICE_JANUS > 20 degree) & (Distance of GANYMEDE from JUICE_JANUS < 50000.0 kilometer))
Restructured: ((Phase Angle of GANYMEDE with respect to SUN as seen from JUICE_JANUS > 20 degree) & (Distance of GANYMEDE from JUICE_JANUS < 50000.0 kilometer))

Types match: True
Left types match: True
Right types match: True


## Alternative: Using convenience functions

The serialization module also provides convenience functions that create the converter automatically:

In [ ]:
from spice_segmenter import unstructure_constraint, structure_constraint

# Serialize using convenience function (creates converter internally)
simple_data = unstructure_constraint(cc)
print("Serialized data:")
print(simple_data)

# Deserialize using convenience function
simple_reconstructed = structure_constraint(simple_data)
print("\nReconstructed constraint:")
print(simple_reconstructed)
print(f"\nMatches original: {type(simple_reconstructed) == type(cc)}")

Serialized data:
{'time_step': None, 'left': {'time_step': None, 'left': {'observer': 'JUICE_JANUS', 'target': 'GANYMEDE', 'light_time_correction': 'NONE', 'third_body': 'SUN', 'type': 'PhaseAngle'}, 'right': {'_value': '20 degree', 'type': 'ScalarConstant'}, 'operator': '>', 'type': 'Constraint'}, 'right': {'time_step': None, 'left': {'observer': 'JUICE_JANUS', 'target': 'GANYMEDE', 'light_time_correction': 'NONE', 'type': 'Distance'}, 'right': {'_value': '50000.0 kilometer', 'type': 'ScalarConstant'}, 'operator': '<', 'type': 'Constraint'}, 'operator': '&', 'type': 'Constraint'}

Reconstructed constraint:
((Phase Angle of GANYMEDE with respect to SUN as seen from JUICE_JANUS > 20 degree) & (Distance of GANYMEDE from JUICE_JANUS < 50000.0 kilometer))

Matches original: True

Reconstructed constraint:
((Phase Angle of GANYMEDE with respect to SUN as seen from JUICE_JANUS > 20 degree) & (Distance of GANYMEDE from JUICE_JANUS < 50000.0 kilometer))

Matches original: True


## JSON Serialization

The converter can also work with JSON strings:

In [ ]:
import json

# Serialize to JSON string
json_str = json.dumps(data, indent=2)
print("JSON representation:")
print(json_str[:500] + "..." if len(json_str) > 500 else json_str)

# Deserialize from JSON string
from_json = structure_constraint(json.loads(json_str))
print(f"\nSuccessfully deserialized from JSON: {from_json}")

JSON representation:
{
  "time_step": null,
  "left": {
    "time_step": null,
    "left": {
      "observer": "JUICE_JANUS",
      "target": "GANYMEDE",
      "light_time_correction": "NONE",
      "third_body": "SUN",
      "type": "PhaseAngle"
    },
    "right": {
      "_value": "20 degree",
      "type": "ScalarConstant"
    },
    "operator": ">",
    "type": "Constraint"
  },
  "right": {
    "time_step": null,
    "left": {
      "observer": "JUICE_JANUS",
      "target": "GANYMEDE",
      "light_time_corre...

Successfully deserialized from JSON: ((Phase Angle of GANYMEDE with respect to SUN as seen from JUICE_JANUS > 20 degree) & (Distance of GANYMEDE from JUICE_JANUS < 50000.0 kilometer))


In [ ]:
dir(c)

['_',
 '__abstractmethods__',
 '__and__',
 '__annotations__',
 '__attrs_attrs__',
 '__attrs_own_setattr__',
 '__attrs_post_init__',
 '__attrs_types_resolved__',
 '__call__',
 '__class__',
 '__delattr__',
 '__dir__',
 '__doc__',
 '__eq__',
 '__firstlineno__',
 '__format__',
 '__ge__',
 '__getattribute__',
 '__getstate__',
 '__gt__',
 '__hash__',
 '__init__',
 '__init_subclass__',
 '__invert__',
 '__le__',
 '__lt__',
 '__match_args__',
 '__module__',
 '__ne__',
 '__new__',
 '__or__',
 '__reduce__',
 '__reduce_ex__',
 '__replace__',
 '__repr__',
 '__setattr__',
 '__setstate__',
 '__sizeof__',
 '__slots__',
 '__static_attributes__',
 '__str__',
 '__subclasshook__',
 '__weakref__',
 '_abc_impl',
 '_handle_other_operand',
 'as_unit',
 'compute_as_spice_function',
 'config',
 'ctype',
 'from_json',
 'has_unit',
 'is_decreasing',
 'is_decreasing_as_spice_function',
 'left',
 'name',
 'operator',
 'render_tree',
 'render_tree_str',
 'right',
 'solve',
 'time_step',
 'to_json',
 'tree',
 'type',